In [1]:
# import current working diretories

import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research


In [2]:
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction")

In [3]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction\\research'

In [4]:
os.chdir("../") 

In [5]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction'

In [6]:
import box
print(box.__version__)

7.4.1


In [7]:
# entity

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:

    root_dir: Path

    # Transformed data paths
    train_data_path: Path  
    test_data_path: Path

    # Path to save trained model
    trained_model_path: Path

    # Path to save model comparison/evaluation report
    model_report_path: Path

In [8]:
from Hotel_Booking_Cancellation_Prediction.constant import *
from Hotel_Booking_Cancellation_Prediction.utils.common import read_yaml, create_directories
from Hotel_Booking_Cancellation_Prediction.entity.config_entity import ModelTrainerConfig
from Hotel_Booking_Cancellation_Prediction.config.configuration import ConfigurationManager


In [9]:
# configuration manager

def get_model_trainer_config(self) -> ModelTrainerConfig:

    config = self.config["model_trainer"]

    create_directories([config.root_dir])

    model_trainer_config = ModelTrainerConfig(

        root_dir=Path(config.root_dir),
        train_data_path=Path(config.train_data_path),
        test_data_path=Path(config.test_data_path),
        trained_model_path=Path(config.trained_model_path),
        model_report_path=Path(config.model_report_path)
    )
    return model_trainer_config


In [10]:
# import all libraries

import os
import pandas as pd
import joblib
import numpy as np

from Hotel_Booking_Cancellation_Prediction.logging import logger

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)

In [11]:
# components


class ModelTrainer:

    def __init__(self, config):

        self.config = config

    # LOAD TRANSFORMED DATA

    def load_data(self):

        # Scaled data
        train_scaled = pd.read_csv(
            self.config.train_data_path
        )

        test_scaled = pd.read_csv(
            self.config.test_data_path
        )

        # Unscaled data
        unscaled_train_path = os.path.join(
            os.path.dirname(
                self.config.train_data_path
            ),
            "train_unscaled.csv"
        )

        unscaled_test_path = os.path.join(
            os.path.dirname(
                self.config.test_data_path
            ),
            "test_unscaled.csv"
        )

        train_unscaled = pd.read_csv(
            unscaled_train_path
        )

        test_unscaled = pd.read_csv(
            unscaled_test_path
        )

        return (
            train_scaled,
            test_scaled,
            train_unscaled,
            test_unscaled
        )

    # CALCULATE CLASSIFICATION METRICS

    def evaluate_model(self, model, X_test, y_test):

        # Predictions

        y_pred = model.predict(
            X_test
        )

        # Probability predictions

        if hasattr(
            model,
            "predict_proba"
        ):

            y_prob = model.predict_proba(
                X_test
            )[:, 1]

        else:

            y_score = model.decision_function(
                X_test
            )

            y_prob = (
                1 /
                (
                    1 +
                    np.exp(
                        -y_score
                    )
                )
            )

        # Confusion matrix

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            y_pred
        ).ravel()

        # Eight metrics

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        pr_auc = average_precision_score(
            y_test,
            y_prob
        )

        loss = log_loss(
            y_test,
            y_prob
        )

        return {
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "Specificity": specificity,
            "F1 Score": f1,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "Log Loss": loss
        }

    # TRAIN MODELS

    def train_models(self):

        (
            train_scaled,
            test_scaled,
            train_unscaled,
            test_unscaled
        ) = self.load_data()

        # SCALED DATA

        X_train_scaled = train_scaled.drop(
            columns=["is_canceled"]
        )

        y_train_scaled = train_scaled[
            "is_canceled"
        ]

        X_test_scaled = test_scaled.drop(
            columns=["is_canceled"]
        )

        y_test_scaled = test_scaled[
            "is_canceled"
        ]

        # UNSCALED DATA

        X_train_unscaled = train_unscaled.drop(
            columns=["is_canceled"]
        )

        y_train_unscaled = train_unscaled[
            "is_canceled"
        ]

        X_test_unscaled = test_unscaled.drop(
            columns=["is_canceled"]
        )

        y_test_unscaled = test_unscaled[
            "is_canceled"
        ]

        # FINAL MODEL

        # choose the final parameters from all the model from model_experiments

        models = {

            "Logistic Regression": (
                LogisticRegression(
                    C=100,
                    class_weight="balanced",
                    penalty="l1",
                    solver="liblinear",
                    max_iter=1000,
                    random_state=42
                ),
                "scaled"
            ),

            "Decision Tree": (
                DecisionTreeClassifier(
                    criterion="gini",
                    max_depth=20,
                    min_samples_leaf=1,
                    min_samples_split=2,
                    random_state=42
                ),
                "unscaled"
            ),

            "Random Forest": (
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=None,
                    max_features=None,
                    min_samples_leaf=1,
                    min_samples_split=2,
                    random_state=42,
                    n_jobs=-1
                ),
                "unscaled"
            ),

            "SVM": (
                LinearSVC(
                    C=0.1,
                    class_weight="balanced",
                    random_state=42,
                    max_iter=5000
                ),
                "scaled"
            ),

            "KNN": (
                KNeighborsClassifier(
                    n_neighbors=11,
                    weights="distance"
                ),
                "scaled"
            ),

            "Naive Bayes": (
                GaussianNB(
                    var_smoothing=0.01
                ),
                "scaled"
            )
        }

        results = []

        trained_models = {}

        # TRAIN EACH MODEL

        for model_name, (model, data_type) in models.items():

            print(f"Training {model_name}...")

            if data_type == "scaled":

                X_train = X_train_scaled
                y_train = y_train_scaled

                X_test = X_test_scaled
                y_test = y_test_scaled

            else:

                X_train = X_train_unscaled
                y_train = y_train_unscaled

                X_test = X_test_unscaled
                y_test = y_test_unscaled

            # Train

            model.fit(X_train,y_train)

            # Evaluate

            metrics = self.evaluate_model(model,X_test,y_test)

            metrics["Model"] = model_name

            results.append(metrics)

            trained_models[model_name] = model

        # MODEL COMPARISON

        results_df = pd.DataFrame(results)

        results_df = results_df[
            [
                "Model",
                "Accuracy",
                "Precision",
                "Recall",
                "Specificity",
                "F1 Score",
                "ROC-AUC",
                "PR-AUC",
                "Log Loss"
            ]
        ]

        # SELECT BEST MODEL

        best_model_name = (
            results_df
            .sort_values(
                by="F1 Score",
                ascending=False
            )
            .iloc[0]["Model"]
        )

        best_model = trained_models[
            best_model_name
        ]

        print("\nModel Comparison:")

        print(results_df.round(4))

        print("\nBest Model:")

        print(best_model_name)

        # SAVE MODEL REPORT

        results_df.to_csv(
            self.config.model_report_path,
            index=False
        )

        # SAVE BEST MODEL

        joblib.dump(
            best_model,
            self.config.trained_model_path
        )

        print("\nModel Trainer Completed")

        print("Model Report Saved At:",
            self.config.model_report_path
        )

        print("Best Model Saved At:",
            self.config.trained_model_path
        )

        return (results_df,
            best_model_name
        )

In [12]:
# pipeline

class ModelTrainerTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        try:

            logger.info(">>>>>> Model Trainer Stage Started <<<<<<")

            config = ConfigurationManager()

            model_trainer_config = (config.get_model_trainer_config() )

            model_trainer = ModelTrainer(config=model_trainer_config)

            model_trainer.train_models()

            logger.info(
                ">>>>>> Model Trainer Stage Completed <<<<<<"
            )

        except Exception as e:

            logger.exception(e)
            raise e

In [13]:
obj = ModelTrainerTrainingPipeline()
obj.main()

[2026-08-16 23:46:53,250: INFO: 2234383976: >>>>>> Model Trainer Stage Started <<<<<<]
[2026-08-16 23:46:53,262: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\config\config.yaml loaded successfully]
[2026-08-16 23:46:53,271: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\params.yaml loaded successfully]
[2026-08-16 23:46:53,275: INFO: common: created directory at artifacts]
[2026-08-16 23:46:53,275: INFO: common: created directory at artifacts/model_trainer]
Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training SVM...
Training KNN...
Training Naive Bayes...

Model Comparison:
                 Model  Accuracy  Precision  Recall  Specificity  F1 Score  \
0  Logistic Regression    0.8277     0.7416  0.8210       0.8316    0.7793   
1        Decision Tree    0.8680     0.8162  0.8306       0.8899    0.8233   
2        Random For